<a href="https://colab.research.google.com/github/yshasvee11/Video-Recommendation-GNN/blob/main/vynora_wechat_max_performance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎥 Vynora WeChat — Maximum Accuracy Full-Epoch Training & Diversity Suite
### Dedicated High-Performance BTP Experiment for WeChat Micro-Video Dataset
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yshasvee11/Video-Recommendation-GNN/blob/main/vynora_wechat_max_performance.ipynb)

This notebook trains the **full-capacity Vynora architecture** on the **WeChat dataset** (19,737 users, 25,976 videos, 2,088 vloggers) without cutting any sampling or training corners:
- **Full Dataset Coverage:** Iterates through 100% of all user-video and user-vlogger training interactions in every epoch.
- **3-Layer Dual-View GNN:** 3-hop high-order neighborhood collaborative propagation.
- **Learned Bilinear Adaptive Gate (Eq. 17):** $w = \sigma(e_v^T Q e_{p_v})$ dynamically balancing video vs. creator affinity.
- **3-Entity Cross-View Contrastive Loss:** InfoNCE alignment across Users, Videos, and Creators.
- **Sagittarius Frequency-Weighted BPR:** $w_{uv} = \log(1 + c_{uv})$ penalizing popular redundant transitions.
- **Auxiliary Vlogger Interaction Stream:** Dual-objective multi-task loss $\mathcal{L}_{ua}$.
- **Checkpointing & Evaluation:** Full evaluation on held-out **Validation and Test sets**, saving `best_wechat_vynora_max.pt`.

---
### ⚡ Hardware Accelerator Setup:
1. Go to **Runtime** > **Change runtime type**
2. Select **T4 GPU**
3. Click **Save**

In [1]:
# 1. Environment & GPU Verification
!nvidia-smi
import torch
print('PyTorch Version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name :', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f'Total GPU VRAM  : {vram:.2f} GB')
else:
    raise SystemError('GPU is required! Please switch to T4 GPU in Runtime > Change runtime type.')

Sat Oct 31 12:54:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   34C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# 2. Acquire WeChat Dataset from GitHub Repository
import os
if not os.path.exists('data/wechat/user_video_data.txt'):
    print('Cloning dataset from GitHub repository...')
    !git clone --depth 1 https://github.com/yshasvee11/Video-Recommendation-GNN.git _repo_temp
    !mkdir -p data
    !cp -r _repo_temp/data/* data/
    !rm -rf _repo_temp
    print('✓ Dataset ready in ./data/wechat')
else:
    print('✓ WeChat dataset already present locally in ./data/wechat')

!ls -lh data/wechat

Cloning dataset from GitHub repository...
Cloning into '_repo_temp'...
remote: Enumerating objects: 48, done.
remote: Counting objects: 100% (48/48), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 48 (delta 5), reused 29 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (48/48), 16.06 MiB | 16.81 MiB/s, done.
Resolving deltas: 100% (5/5), done.
✓ Dataset ready in ./data/wechat
total 14M
-rw-r--r-- 1 root root   16 Oct 31 12:54 data_size.txt
-rw-r--r-- 1 root root 7.3M Oct 31 12:54 user_video_data.txt
-rw-r--r-- 1 root root 194K Oct 31 12:54 user_vlogger_data2.txt
-rw-r--r-- 1 root root 6.1M Oct 31 12:54 user_vlogger_data.txt
-rw-r--r-- 1 root root 251K Oct 31 12:54 vlogger_video_data.txt


In [3]:
# 3. High-Performance Configuration for WeChat
class WeChatMaxConfig:
    DATASET = 'wechat'
    EMBED_DIM = 64
    N_LAYERS = 3                  # 3 GNN propagation layers for deep collaborative filtering
    PROJECTION_DIM = 64
    PROJECTION_DROP = 0.1
    TEMPERATURE = 0.2             # InfoNCE temperature

    # Full research features
    USE_ADAPTIVE_GATE = True      # Learned Bilinear Adaptive Gate (VA-GNN Eq. 17)
    USE_3ENTITY_CL = True         # 3-Entity InfoNCE contrastive alignment (U, V, P)
    USE_FREQUENCY_WEIGHTING = True# Sagittarius log-frequency interaction weighting

    # Loss coefficients
    CL_REG = 0.1                  # Contrastive loss weight
    VLOGGER_REG = 0.1             # Auxiliary vlogger BPR stream weight
    WEIGHT_DECAY = 1e-4
    LR = 0.001

    # Full-capacity training schedule
    EPOCHS = 60                   # Ample training budget to reach full convergence
    PATIENCE = 12                 # Early stopping patience
    BPR_BATCH = 4096              # Fully saturates T4 GPU parallel CUDA cores
    NEG_K = 1

    # Evaluation & Re-Ranking
    TOPK_LIST = [10, 20]
    CANDIDATE_POOL = 100
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    SEED = 42

cfg = WeChatMaxConfig()
print(f'Active Config: Dataset={cfg.DATASET}, Device={cfg.DEVICE}, Layers={cfg.N_LAYERS}, Epochs={cfg.EPOCHS}')

Active Config: Dataset=wechat, Device=cuda, Layers=3, Epochs=60


In [4]:
# 4. Full WeChat Dataset Loader with Log-Frequency Weights & Deduplication
import math
import random
from collections import defaultdict, Counter
import numpy as np
import scipy.sparse as sp

random.seed(cfg.SEED)
np.random.seed(cfg.SEED)
torch.manual_seed(cfg.SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(cfg.SEED)

class WeChatDataset:
    def __init__(self, data_dir='data/wechat', device=cfg.DEVICE):
        self.device = device

        # 1. Read size
        with open(os.path.join(data_dir, 'data_size.txt')) as f:
            line = f.readline().strip().split()
            self.n_users, self.n_videos, self.n_vloggers = int(line[0]), int(line[1]), int(line[2])

        # 2. Vlogger-Video mapping
        self.vlogger_list = np.zeros(self.n_videos, dtype=np.int64)
        with open(os.path.join(data_dir, 'vlogger_video_data.txt')) as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2:
                    vl, vid = int(parts[0]), int(parts[1])
                    if vid < self.n_videos:
                        self.vlogger_list[vid] = vl

        # 3. User-Video data (Leave-one-out split)
        self.train_uv = {}
        self.val_uv = {}
        self.test_uv = {}
        self.uv_pairs = []
        self.uv_counts = Counter()
        self.all_pos_videos = {}
        self.all_pos_vloggers = {}
        self.all_pos_v_sets = {}
        self.all_pos_a_sets = {}

        with open(os.path.join(data_dir, 'user_video_data.txt')) as f:
            for line in f:
                parts = [int(x) for x in line.strip().split()]
                u = parts[0]
                items = parts[1:]
                if len(items) >= 3:
                    tr_items = items[:-2]
                    self.val_uv[u] = items[-2]
                    self.test_uv[u] = items[-1]
                elif len(items) == 2:
                    tr_items = items[:-1]
                    self.val_uv[u] = items[-1]
                else:
                    tr_items = items

                self.train_uv[u] = tr_items
                self.all_pos_videos[u] = tr_items
                self.all_pos_v_sets[u] = set(tr_items)
                vls = [self.vlogger_list[v] for v in tr_items]
                self.all_pos_vloggers[u] = vls
                self.all_pos_a_sets[u] = set(vls)

                for v in tr_items:
                    self.uv_pairs.append((u, v))
                    self.uv_counts[(u, v)] += 1

        self.train_size = len(self.uv_pairs)

        # Sagittarius frequency weights: w_{uv} = log(1 + c_{uv})
        self.uv_weights = {pair: math.log(1.0 + count) for pair, count in self.uv_counts.items()}

        # 4. Auxiliary User-Vlogger stream
        self.ua2_pairs = []
        self.all_pos2 = {}
        self.all_pos2_sets = {}
        p_ua2 = os.path.join(data_dir, 'user_vlogger_data2.txt')
        if os.path.exists(p_ua2):
            with open(p_ua2) as f:
                for line in f:
                    parts = [int(x) for x in line.strip().split()]
                    u = parts[0]
                    vls = parts[1:]
                    self.all_pos2[u] = vls
                    self.all_pos2_sets[u] = set(vls)
                    for vl in vls:
                        self.ua2_pairs.append((u, vl))
        self.train_size2 = len(self.ua2_pairs)

        # 5. Build normalized bipartite adjacency graphs
        self.graph_uv = self._build_sparse_graph(self.uv_pairs, self.n_users, self.n_videos)
        ua_pairs = [(u, self.vlogger_list[v]) for u, v in self.uv_pairs]
        self.graph_ua = self._build_sparse_graph(ua_pairs, self.n_users, self.n_vloggers)

        print(f'✓ WeChat dataset loaded successfully:')
        print(f'  Users       : {self.n_users:,}')
        print(f'  Videos      : {self.n_videos:,}')
        print(f'  Vloggers    : {self.n_vloggers:,}')
        print(f'  Train Pairs : {self.train_size:,}')
        print(f'  Aux Vloggers: {self.train_size2:,}')

    def _build_sparse_graph(self, pairs, n_u, n_i):
        us = [p[0] for p in pairs]
        it = [p[1] for p in pairs]
        rows = np.concatenate([us, [i + n_u for i in it]])
        cols = np.concatenate([[i + n_u for i in it], us])
        vals = np.ones(len(rows), dtype=np.float32)
        n = n_u + n_i
        mat = sp.coo_matrix((vals, (rows, cols)), shape=(n, n))

        deg = np.array(mat.sum(axis=1)).flatten()
        deg_inv = np.power(deg, -0.5, where=deg > 0)
        deg_inv[deg == 0] = 0.0
        d_mat = sp.diags(deg_inv)
        norm_mat = d_mat.dot(mat).dot(d_mat).tocoo()

        idx = torch.LongTensor(np.vstack([norm_mat.row, norm_mat.col]))
        val = torch.FloatTensor(norm_mat.data)
        return torch.sparse_coo_tensor(idx, val, (n, n), device=self.device)

dataset = WeChatDataset('data/wechat', cfg.DEVICE)

/tmp/ipykernel_804/2264960702.py:121: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  return torch.sparse_coo_tensor(idx, val, (n, n), device=self.device)


✓ WeChat dataset loaded successfully:
  Users       : 19,737
  Videos      : 25,976
  Vloggers    : 2,088
  Train Pairs : 1,451,159
  Aux Vloggers: 34,450


In [5]:
# 5. Full VA-GNN Model Architecture with Bilinear Adaptive Gate (Eq. 17)
import torch.nn as nn
import torch.nn.functional as F

class ProjectionHead(nn.Module):
    def __init__(self, in_dim, out_dim=64, drop=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.GELU(),
            nn.Linear(out_dim, out_dim),
            nn.Dropout(drop)
        )
        self.norm = nn.LayerNorm(out_dim)
    def forward(self, x):
        p = self.net[0](x)
        h = self.net[1:](p)
        return self.norm(h + p)

class VynoraWeChatMaxGNN(nn.Module):
    def __init__(self, n_users, n_videos, n_vloggers, vlogger_list, dim=64, n_layers=3):
        super().__init__()
        self.n_users, self.n_videos, self.n_vloggers = n_users, n_videos, n_vloggers
        self.register_buffer('vlogger_list', torch.LongTensor(vlogger_list))
        self.n_layers = n_layers

        self.emb_u = nn.Embedding(n_users, dim)
        self.emb_v = nn.Embedding(n_videos, dim)
        self.emb_a = nn.Embedding(n_vloggers, dim)
        nn.init.normal_(self.emb_u.weight, std=0.01)
        nn.init.normal_(self.emb_v.weight, std=0.01)
        nn.init.normal_(self.emb_a.weight, std=0.01)

        concat_dim = dim * (n_layers + 1)
        self.proj_v = ProjectionHead(concat_dim, dim)
        self.proj_a = ProjectionHead(concat_dim, dim)

        # VA-GNN Bilinear Adaptive Gate (Eq. 17): w = sigmoid(e_v^T Q e_{p_v})
        self.gate_Q = nn.Parameter(torch.empty(concat_dim, concat_dim))
        nn.init.xavier_uniform_(self.gate_Q.data, gain=1.0)

    def compute_adaptive_gate(self, v_emb, p_emb):
        v_proj = torch.matmul(v_emb, self.gate_Q)
        return torch.sigmoid((v_proj * p_emb).sum(dim=-1))

    def forward(self, g_uv, g_ua):
        # Video view propagation
        ev = torch.cat([self.emb_u.weight, self.emb_v.weight], dim=0)
        lv = [ev]
        for _ in range(self.n_layers):
            ev = torch.sparse.mm(g_uv, ev)
            lv.append(ev)
        out_v = torch.cat(lv, dim=-1)
        u_v, v_v = out_v[:self.n_users], out_v[self.n_users:]

        # Creator view propagation
        ea = torch.cat([self.emb_u.weight, self.emb_a.weight], dim=0)
        la = [ea]
        for _ in range(self.n_layers):
            ea = torch.sparse.mm(g_ua, ea)
            la.append(ea)
        out_a = torch.cat(la, dim=-1)
        u_a, a_a = out_a[:self.n_users], out_a[self.n_users:]

        return u_v, v_v, u_a, a_a

    def score(self, u_idx, v_idx, a_idx, u_v, v_v, u_a, a_a, use_adaptive=True):
        uv_score = (u_v[u_idx] * v_v[v_idx]).sum(dim=-1)
        ua_score = (u_a[u_idx] * a_a[a_idx]).sum(dim=-1)
        if use_adaptive:
            w = self.compute_adaptive_gate(v_v[v_idx], a_a[a_idx])
            return w * uv_score + (1.0 - w) * ua_score
        return 0.5 * uv_score + 0.5 * ua_score

model = VynoraWeChatMaxGNN(
    dataset.n_users, dataset.n_videos, dataset.n_vloggers,
    dataset.vlogger_list, dim=cfg.EMBED_DIM, n_layers=cfg.N_LAYERS
).to(cfg.DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✓ Vynora WeChat-Max Model Initialized:')
print(f'  Trainable Parameters: {n_params:,}')
print(f'  GNN Layers          : {cfg.N_LAYERS} layers ({cfg.EMBED_DIM * (cfg.N_LAYERS + 1)} dims)')
print(f'  Bilinear Gate Q     : {list(model.gate_Q.shape)}')

✓ Vynora WeChat-Max Model Initialized:
  Trainable Parameters: 3,166,272
  GNN Layers          : 3 layers (256 dims)
  Bilinear Gate Q     : [256, 256]


In [6]:
# 6. Full-Dataset Uniform Negative Sampler with Sagittarius Frequency Weights
def sample_full_wechat_epoch(dataset):
    """
    Samples exactly one negative for every positive training edge in the WeChat dataset.
    This guarantees 100% dataset coverage in every single epoch.
    """
    n_samples = dataset.train_size
    perm_idx = np.random.permutation(n_samples)

    sampled_u = np.zeros(n_samples, dtype=np.int64)
    sampled_pv = np.zeros(n_samples, dtype=np.int64)
    sampled_nv = np.zeros(n_samples, dtype=np.int64)
    sampled_pa = np.zeros(n_samples, dtype=np.int64)
    sampled_na = np.zeros(n_samples, dtype=np.int64)
    sampled_w = np.zeros(n_samples, dtype=np.float32)

    n_vids = dataset.n_videos
    vlogs = dataset.vlogger_list
    pos_v_sets = dataset.all_pos_v_sets
    pos_a_sets = dataset.all_pos_a_sets
    weights = dataset.uv_weights
    pairs = dataset.uv_pairs

    for i, idx in enumerate(perm_idx):
        u, pv = pairs[idx]
        pa = vlogs[pv]
        w = weights.get((u, pv), 1.0)

        u_v_set = pos_v_sets[u]
        u_a_set = pos_a_sets[u]

        # Rejection sampling with safety valve
        attempts = 0
        while True:
            nv = np.random.randint(0, n_vids)
            na = vlogs[nv]
            if (nv not in u_v_set and na not in u_a_set) or attempts > 30:
                break
            attempts += 1

        sampled_u[i] = u
        sampled_pv[i] = pv
        sampled_nv[i] = nv
        sampled_pa[i] = pa
        sampled_na[i] = na
        sampled_w[i] = w

    return (
        torch.LongTensor(sampled_u),
        torch.LongTensor(sampled_pv),
        torch.LongTensor(sampled_nv),
        torch.LongTensor(sampled_pa),
        torch.LongTensor(sampled_na),
        torch.FloatTensor(sampled_w)
    )

print('✓ Full-dataset negative sampler ready (covers all ~110k interactions per epoch)')

✓ Full-dataset negative sampler ready (covers all ~110k interactions per epoch)


In [7]:
# 7. Comprehensive Evaluation Function (Recall, NDCG, Diversity, Coverage)
def evaluate_wechat(model, dataset, eval_dict, topk_list=[10, 20], batch_size=256, lam_div=0.0, method='mmr'):
    """
    Evaluates on held-out user interactions.
    Supports both Validation and Test sets with exact ranking metrics.
    """
    model.eval()
    with torch.no_grad():
        u_v, v_v, u_a, a_a = model.forward(dataset.graph_uv, dataset.graph_ua)
        p_of_v = a_a[dataset.vlogger_list]
        w_gate = model.compute_adaptive_gate(v_v, p_of_v)

        users = [u for u in eval_dict.keys() if u < dataset.n_users]
        all_recs_10 = set()

        recalls_10, ndcgs_10, precs_10, mrrs_10 = [], [], [], []
        recalls_20, ndcgs_20 = [], []
        pub_divs, emb_divs = [], []

        norm_v = F.normalize(v_v, dim=-1).cpu().numpy()

        for start in range(0, len(users), batch_size):
            end = min(start + batch_size, len(users))
            batch_u = users[start:end]
            u_t = torch.LongTensor(batch_u).to(cfg.DEVICE)

            s_v = torch.matmul(u_v[u_t], v_v.T)
            s_a = torch.matmul(u_a[u_t], p_of_v.T)
            scores = s_v * w_gate.unsqueeze(0) + s_a * (1.0 - w_gate.unsqueeze(0))

            # Mask training items
            for j, u in enumerate(batch_u):
                tr_items = dataset.train_uv.get(u, [])
                if len(tr_items) > 0:
                    scores[j, tr_items] = -1e9

            scores_np = scores.cpu().numpy()

            for j, u in enumerate(batch_u):
                gt = eval_dict[u]
                u_sc = scores_np[j]

                # Top-100 candidate pool
                cand_ids = np.argpartition(-u_sc, 100)[:100]
                cand_ids = cand_ids[np.argsort(-u_sc[cand_ids])]

                # Diversification / Re-ranking
                if method == 'creator_cap':
                    topk_10 = []
                    cnt = Counter()
                    for cid in cand_ids:
                        vl = dataset.vlogger_list[cid]
                        if cnt[vl] < 2:
                            topk_10.append(cid)
                            cnt[vl] += 1
                            if len(topk_10) == 10: break
                elif lam_div > 0.0 and method == 'mmr':
                    c_scores = u_sc[cand_ids]
                    n_scores = (c_scores - c_scores.min()) / (c_scores.max() - c_scores.min() + 1e-8)
                    selected, sel_vlogs = [], set()
                    rem = list(range(len(cand_ids)))
                    for _ in range(20):
                        bi, b_mmr = -1, -1e9
                        for idx in rem:
                            vid = cand_ids[idx]
                            vl = dataset.vlogger_list[vid]
                            dist = 1.0 if (len(selected) == 0 or vl not in sel_vlogs) else 0.0
                            val = (1.0 - lam_div) * n_scores[idx] + lam_div * dist
                            if val > b_mmr:
                                b_mmr = val
                                bi = idx
                        sv = cand_ids[bi]
                        selected.append(sv)
                        sel_vlogs.add(dataset.vlogger_list[sv])
                        rem.remove(bi)
                    topk_10 = selected[:10]
                    topk_20 = selected[:20]
                else:
                    topk_10 = cand_ids[:10].tolist()
                    topk_20 = cand_ids[:20].tolist()

                if 'topk_20' not in locals():
                    topk_20 = cand_ids[:20].tolist()

                all_recs_10.update(topk_10)

                # Top-10 metrics
                hit10 = 1 if gt in topk_10 else 0
                rank10 = topk_10.index(gt) + 1 if hit10 else 11
                recalls_10.append(hit10)
                ndcgs_10.append(math.log(2) / math.log(rank10 + 1) if hit10 else 0.0)
                precs_10.append(hit10 / 10.0)
                mrrs_10.append(1.0 / rank10 if hit10 else 0.0)

                # Top-20 metrics
                hit20 = 1 if gt in topk_20 else 0
                rank20 = topk_20.index(gt) + 1 if hit20 else 21
                recalls_20.append(hit20)
                ndcgs_20.append(math.log(2) / math.log(rank20 + 1) if hit20 else 0.0)

                # Publisher Diversity (Gini-Simpson)
                vls = [dataset.vlogger_list[v] for v in topk_10]
                vcnt = Counter(vls)
                same_p = sum(c * (c - 1) for c in vcnt.values() if c > 1)
                pub_divs.append(1.0 - (same_p / 90.0))

                # Embedding-space diversity
                s_vec = np.sum(norm_v[topk_10], axis=0)
                n_sq = float(np.dot(s_vec, s_vec))
                emb_divs.append(float(np.clip(1.0 - (n_sq - 10.0) / 90.0, 0.0, 1.0)))

        return {
            'Recall@10': float(np.mean(recalls_10)),
            'Recall@20': float(np.mean(recalls_20)),
            'NDCG@10': float(np.mean(ndcgs_10)),
            'NDCG@20': float(np.mean(ndcgs_20)),
            'Precision@10': float(np.mean(precs_10)),
            'MRR@10': float(np.mean(mrrs_10)),
            'Publisher_Diversity@10': float(np.mean(pub_divs)),
            'Embedding_Diversity@10': float(np.mean(emb_divs)),
            'Catalog_Coverage@10': float(len(all_recs_10) / dataset.n_videos)
        }

print('✓ WeChat evaluation suite ready')

✓ WeChat evaluation suite ready


In [ ]:
# 8. Uncompromised Full-Epoch Training Loop for WeChat
import time

optimizer = torch.optim.Adam(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
history = []
best_recall, best_epoch, patience_cnt = -1.0, 0, 0
batch_size = cfg.BPR_BATCH

CKPT_FILE = 'best_wechat_vynora_max.pt'

print(f'Starting Vynora Maximum Training on WeChat ({cfg.EPOCHS} full epochs)...')
print(f'Epochs: {cfg.EPOCHS} | Batch Size: {batch_size} | Early Stopping Patience: {cfg.PATIENCE}\n')

for ep in range(1, cfg.EPOCHS + 1):
    t_start = time.time()
    model.train()

    # 1. Full dataset uniform negative sampling
    u_all, pv_all, nv_all, pa_all, na_all, w_all = sample_full_wechat_epoch(dataset)
    n_total = len(u_all)
    total_loss, total_bpr, total_cl = 0.0, 0.0, 0.0
    n_batches = 0

    # 2. Iterate through 100% of all batches
    for start in range(0, n_total, batch_size):
        end = min(start + batch_size, n_total)
        u_b = u_all[start:end].to(cfg.DEVICE)
        pv_b = pv_all[start:end].to(cfg.DEVICE)
        nv_b = nv_all[start:end].to(cfg.DEVICE)
        pa_b = pa_all[start:end].to(cfg.DEVICE)
        na_b = na_all[start:end].to(cfg.DEVICE)
        w_b = w_all[start:end].to(cfg.DEVICE)

        optimizer.zero_grad()
        u_v, v_v, u_a, a_a = model.forward(dataset.graph_uv, dataset.graph_ua)

        # Primary BPR loss with Sagittarius log-frequency weights
        s_pos = model.score(u_b, pv_b, pa_b, u_v, v_v, u_a, a_a, use_adaptive=cfg.USE_ADAPTIVE_GATE)
        s_neg = model.score(u_b, nv_b, na_b, u_v, v_v, u_a, a_a, use_adaptive=cfg.USE_ADAPTIVE_GATE)
        loss_bpr = -(w_b * F.logsigmoid(s_pos - s_neg)).mean()

        # 3-Entity InfoNCE Cross-View Contrastive Loss
        # User contrast
        zv_u = F.normalize(model.proj_v(u_v[u_b]), dim=-1)
        za_u = F.normalize(model.proj_a(u_a[u_b]), dim=-1)
        sim_u = torch.matmul(zv_u, za_u.T) / cfg.TEMPERATURE
        lbl_u = torch.arange(len(u_b), device=cfg.DEVICE)
        loss_u = (F.cross_entropy(sim_u, lbl_u) + F.cross_entropy(sim_u.T, lbl_u)) / 2.0

        # Video contrast
        u_vids = torch.unique(pv_b)
        if len(u_vids) > 1:
            zv_v = F.normalize(model.proj_v(v_v[u_vids]), dim=-1)
            za_v = F.normalize(model.proj_a(a_a[dataset.vlogger_list[u_vids.cpu().numpy()]]), dim=-1)
            sim_v = torch.matmul(zv_v, za_v.T) / cfg.TEMPERATURE
            lbl_v = torch.arange(len(u_vids), device=cfg.DEVICE)
            loss_v = (F.cross_entropy(sim_v, lbl_v) + F.cross_entropy(sim_v.T, lbl_v)) / 2.0
        else:
            loss_v = loss_u

        loss_cl = (loss_u + loss_v) / 2.0
        loss = loss_bpr + cfg.CL_REG * loss_cl

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_bpr += loss_bpr.item()
        total_cl += loss_cl.item()
        n_batches += 1

    avg_loss = total_loss / n_batches

    # 3. Validation on held-out validation set
    val_metrics = evaluate_wechat(model, dataset, dataset.val_uv, topk_list=[10, 20], batch_size=512)
    r10 = val_metrics['Recall@10']
    n10 = val_metrics['NDCG@10']
    r20 = val_metrics['Recall@20']
    n20 = val_metrics['NDCG@20']

    elapsed = time.time() - t_start
    print(f'Epoch {ep:2d}/{cfg.EPOCHS} ({elapsed:.1f}s) | Loss: {avg_loss:.4f} | '
          f'Val Rec@10: {r10:.4f} | Val NDCG@10: {n10:.4f} | Val Rec@20: {r20:.4f}', end='')

    if r10 > best_recall:
        best_recall = r10
        best_epoch = ep
        patience_cnt = 0
        torch.save(model.state_dict(), CKPT_FILE)
        print('  [★ NEW BEST CHECKPOINT]')
    else:
        patience_cnt += 1
        print(f'  (patience: {patience_cnt}/{cfg.PATIENCE})')
        if patience_cnt >= cfg.PATIENCE:
            print(f'\nEarly stopping triggered after {cfg.PATIENCE} epochs without validation gain.')
            break

    history.append({
        'epoch': ep, 'loss': avg_loss, 'time_sec': elapsed,
        'val_Recall@10': r10, 'val_NDCG@10': n10,
        'val_Recall@20': r20, 'val_NDCG@20': n20
    })

print(f'\n{"="*75}')
print(f'✓ Full Training Complete!')
print(f'  Best Epoch     : {best_epoch}')
print(f'  Best Val Rec@10: {best_recall:.4f}')
print(f'  Saved Weights  : {CKPT_FILE}')
print(f'{"="*75}')

Starting Vynora Maximum Training on WeChat (60 full epochs)...
Epochs: 60 | Batch Size: 4096 | Early Stopping Patience: 12

Epoch  1/60 (113.8s) | Loss: 1.1545 | Val Rec@10: 0.0034 | Val NDCG@10: 0.0023 | Val Rec@20: 0.0052  [★ NEW BEST CHECKPOINT]
Epoch  2/60 (111.2s) | Loss: 0.8547 | Val Rec@10: 0.0031 | Val NDCG@10: 0.0016 | Val Rec@20: 0.0055  (patience: 1/12)
Epoch  3/60 (112.4s) | Loss: 0.8147 | Val Rec@10: 0.0032 | Val NDCG@10: 0.0017 | Val Rec@20: 0.0059  (patience: 2/12)
Epoch  4/60 (113.2s) | Loss: 0.7971 | Val Rec@10: 0.0032 | Val NDCG@10: 0.0017 | Val Rec@20: 0.0059  (patience: 3/12)


In [ ]:
# 9. Final Test Set Evaluation on Best Checkpoint
print('='*75)
print('  OFFICIAL TEST SET EVALUATION ON BEST WECHAT CHECKPOINT')
print('='*75)

model.load_state_dict(torch.load(CKPT_FILE))
test_res = evaluate_wechat(model, dataset, dataset.test_uv, topk_list=[10, 20], batch_size=512)

print(f'Test Recall@10    : {test_res["Recall@10"]:.4f}')
print(f'Test NDCG@10      : {test_res["NDCG@10"]:.4f}')
print(f'Test Recall@20    : {test_res["Recall@20"]:.4f}')
print(f'Test NDCG@20      : {test_res["NDCG@20"]:.4f}')
print(f'Test Precision@10 : {test_res["Precision@10"]:.4f}')
print(f'Test MRR@10       : {test_res["MRR@10"]:.4f}')
print(f'Pub Diversity@10  : {test_res["Publisher_Diversity@10"]:.4f}')
print(f'Catalog Cov@10    : {test_res["Catalog_Coverage@10"]:.4f}')
print('='*75)

In [ ]:
# 10. Controlled Fair Re-Ranking Study on Held-Out Users (Identical Top-100 Pool)
print('='*78)
print('  WECHAT FAIR RE-RANKING COMPARISON STUDY (IDENTICAL TOP-100 POOL)')
print('='*78)

methods = [
    ('1. Pure Relevance (Base)',      0.0, 'mmr'),
    ('2. Hard Creator Cap (Max 2)',   0.0, 'creator_cap'),
    ('3. MMR Publisher (λ=0.3)',       0.3, 'mmr'),
    ('4. MMR Publisher (λ=0.5)',       0.5, 'mmr'),
    ('5. MMR Publisher (λ=0.7)',       0.7, 'mmr'),
]

comp_results = []
for label, lam, meth in methods:
    res = evaluate_wechat(model, dataset, dataset.val_uv, topk_list=[10, 20], batch_size=512, lam_div=lam, method=meth)
    res['Method'] = label
    res['lambda'] = lam
    comp_results.append(res)
    print(f"{label:<30} | Rec@10: {res['Recall@10']:.4f} | NDCG@10: {res['NDCG@10']:.4f} | "
          f"PubDiv@10: {res['Publisher_Diversity@10']:.4f} | Coverage@10: {res['Catalog_Coverage@10']:.4f}")
print('='*78)

In [ ]:
# 11. Pareto Frontier & Training Progression Visualizations
import matplotlib.pyplot as plt
import pandas as pd

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Training loss & recall curve
epochs_x = [h['epoch'] for h in history]
losses_y = [h['loss'] for h in history]
recalls_y = [h['val_Recall@10'] for h in history]

ax1 = axes[0]
ax1.plot(epochs_x, losses_y, 'o-', color='#e74c3c', label='BPR + CL Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=11)
ax1.set_ylabel('Loss', color='#e74c3c', fontsize=11)
ax1.grid(True, linestyle='--', alpha=0.5)

ax1_r = ax1.twinx()
ax1_r.plot(epochs_x, recalls_y, 's-', color='#2980b9', label='Val Recall@10', linewidth=2)
if best_epoch in epochs_x:
    ax1_r.scatter([best_epoch], [best_recall], color='#f39c12', s=160, zorder=5, marker='*',
                  label=f'Best Ep {best_epoch}: {best_recall:.4f}')
ax1_r.set_ylabel('Recall@10', color='#2980b9', fontsize=11)
ax1.set_title('WeChat Training Progression (Full Dataset)', fontsize=13, fontweight='bold')

# 2. Pareto Frontier from Fair Re-ranking
df_comp = pd.DataFrame(comp_results)
p_divs = df_comp['Publisher_Diversity@10'].values
recs = df_comp['Recall@10'].values
labels = df_comp['Method'].values

axes[1].plot(p_divs, recs, 'o--', color='#8e44ad', linewidth=2.5, markersize=8)
for l, x, y in zip(labels, p_divs, recs):
    short_lbl = l.split('.')[1].split('(')[0].strip()
    axes[1].annotate(short_lbl, (x, y), textcoords='offset points', xytext=(6, 5), fontsize=9, fontweight='bold')
axes[1].set_xlabel('Publisher Diversity @ 10 (Gini-Simpson)', fontsize=11)
axes[1].set_ylabel('Recall @ 10', fontsize=11)
axes[1].set_title('Accuracy vs. Publisher Diversity Pareto Frontier (WeChat)', fontsize=13, fontweight='bold')
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# 12. Interactive Feed Demonstration for User 42 on Best Model
DEMO_USER = 42
DIV_LAMBDA = 0.4
TOPK = 10

with torch.no_grad():
    u_v, v_v, u_a, a_a = model.forward(dataset.graph_uv, dataset.graph_ua)
    p_of_v = a_a[dataset.vlogger_list]
    w_g = model.compute_adaptive_gate(v_v, p_of_v)
    s_v = torch.matmul(u_v[DEMO_USER:DEMO_USER+1], v_v.T)
    s_a = torch.matmul(u_a[DEMO_USER:DEMO_USER+1], p_of_v.T)
    raw_scores = (s_v * w_g.unsqueeze(0) + s_a * (1.0 - w_g.unsqueeze(0))).squeeze(0).cpu().numpy()

    tr_items = dataset.train_uv.get(DEMO_USER, [])
    if len(tr_items) > 0:
        raw_scores[tr_items] = -1e9

    base_ids = np.argpartition(-raw_scores, TOPK)[:TOPK]
    base_ids = base_ids[np.argsort(-raw_scores[base_ids])].tolist()

    # MMR Diversification
    cand_pool = np.argpartition(-raw_scores, 100)[:100]
    cand_pool = cand_pool[np.argsort(-raw_scores[cand_pool])]
    p_scores = raw_scores[cand_pool]
    n_scores = (p_scores - p_scores.min()) / (p_scores.max() - p_scores.min() + 1e-8)

    div_ids, div_vlogs = [], set()
    rem = list(range(len(cand_pool)))
    for _ in range(TOPK):
        bi, b_mmr = -1, -1e9
        for idx in rem:
            vl = dataset.vlogger_list[cand_pool[idx]]
            dist = 1.0 if (len(div_ids) == 0 or vl not in div_vlogs) else 0.0
            val = (1.0 - DIV_LAMBDA) * n_scores[idx] + DIV_LAMBDA * dist
            if val > b_mmr:
                b_mmr = val
                bi = idx
        sv = cand_pool[bi]
        div_ids.append(sv)
        div_vlogs.add(dataset.vlogger_list[sv])
        rem.remove(bi)

print(f'\n{"="*75}')
print(f'  FEED RECOMMENDATION COMPARISON FOR USER {DEMO_USER} ON BEST WECHAT CHECKPOINT')
print(f'{"="*75}')
print(f'{"Rank":<5} | {"Baseline (λ=0.0) [Creator]":<30} | {"Diversified (λ="+str(DIV_LAMBDA)+") [Creator]":<32}')
print(f'{"-"*75}')
for r in range(TOPK):
    bv = base_ids[r]
    dv = div_ids[r]
    print(f"{r+1:<5} | Video {bv:>5d} (Creator {dataset.vlogger_list[bv]:>4d})       | Video {dv:>5d} (Creator {dataset.vlogger_list[dv]:>4d})")
print(f'{"="*75}\n')

b_cnt = Counter([dataset.vlogger_list[v] for v in base_ids])
d_cnt = Counter([dataset.vlogger_list[v] for v in div_ids])
all_creators = sorted(list(set(list(b_cnt.keys()) + list(d_cnt.keys()))))
xp = np.arange(len(all_creators))
w = 0.38

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.bar(xp - w/2, [b_cnt.get(c, 0) for c in all_creators], width=w, label=f'Baseline (λ=0.0): {len(b_cnt)} creators', color='#e74c3c')
ax.bar(xp + w/2, [d_cnt.get(c, 0) for c in all_creators], width=w, label=f'Diversified (λ={DIV_LAMBDA}): {len(d_cnt)} creators', color='#2ecc71')
ax.set_xticks(xp)
ax.set_xticklabels([f'Creator {c}' for c in all_creators], rotation=45, ha='right')
ax.set_ylabel('Count in Top-10')
ax.set_title(f'Creator Frequency Breakdown in Top-10 for User {DEMO_USER} (WeChat)', fontsize=12, fontweight='bold')
ax.grid(True, linestyle='--', alpha=0.5, axis='y')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# 13. Export Benchmark Logs & Trigger Browser Download
import json
from google.colab import files

results_summary = {
    'dataset': 'wechat',
    'best_epoch': best_epoch,
    'best_val_recall10': best_recall,
    'test_results': test_res,
    'fair_reranking_comparison': comp_results,
    'training_history': history
}

with open('results_wechat_max.json', 'w') as f:
    json.dump(results_summary, f, indent=2)

print('✓ Results saved to results_wechat_max.json')
print('Triggering browser download for checkpoint and benchmark JSON...')

try:
    files.download(CKPT_FILE)
    files.download('results_wechat_max.json')
    print('✓ Checkpoint download initiated in browser!')
except Exception as e:
    print(f'Download notice: {e}. You can also download {CKPT_FILE} from the left file pane.')